In [ ]:
# python# ============================================================
# # feature_engineering.py
# # ============================================================
# import numpy as np
# import pandas as pd
# import os

# BASE_PATH   = '../data/Train'
# FAULT_TYPES = ['BPFI', 'BPFO', 'BSF', 'Cage']


# # ── 1. RMS 정규화 ─────────────────────────────────────────────
# def add_rms_norm(df):
#     for ch in ['ch1', 'ch2', 'ch3', 'ch4']:
#         df[f'{ch}_rms_norm'] = df[f'{ch}_rms'] / (df['rpm_pred'] / 1000)
#     return df


# # ── 2. 전/후방 결함 지표 ──────────────────────────────────────
# def add_front_rear_features(df):
#     front_fault_cols = [f'ch1_fault_{f}' for f in FAULT_TYPES] + \
#                        [f'ch2_fault_{f}' for f in FAULT_TYPES]
#     rear_fault_cols  = [f'ch3_fault_{f}' for f in FAULT_TYPES] + \
#                        [f'ch4_fault_{f}' for f in FAULT_TYPES]

#     df['front_fault_energy'] = df[front_fault_cols].sum(axis=1)
#     df['rear_fault_energy']  = df[rear_fault_cols].sum(axis=1)

#     df['front_rms'] = (df['ch1_rms_norm'] + df['ch2_rms_norm']) / 2
#     df['rear_rms']  = (df['ch3_rms_norm'] + df['ch4_rms_norm']) / 2

#     df['degradation_location'] = (
#         (df['front_rms'] - df['rear_rms']) /
#         (df['front_rms'] + df['rear_rms'] + 1e-10)
#     )
#     return df


# # ── 3. 수직/축방향 결함 지표 ──────────────────────────────────
# def add_vertical_axial_features(df):
#     vertical_fault_cols = [f'ch1_fault_{f}' for f in FAULT_TYPES] + \
#                           [f'ch3_fault_{f}' for f in FAULT_TYPES]
#     axial_fault_cols    = [f'ch2_fault_{f}' for f in FAULT_TYPES] + \
#                           [f'ch4_fault_{f}' for f in FAULT_TYPES]

#     df['vertical_fault_energy'] = df[vertical_fault_cols].sum(axis=1)
#     df['axial_fault_energy']    = df[axial_fault_cols].sum(axis=1)

#     df['fault_direction'] = (
#         (df['vertical_fault_energy'] - df['axial_fault_energy']) /
#         (df['vertical_fault_energy'] + df['axial_fault_energy'] + 1e-10)
#     )
#     return df


# # ── 4. 초기 결함 강도 지표 (상수) ─────────────────────────────
# def add_initial_fault_intensity(df):
#     intensity_feats = [
#         'ch1_kurtosis',       'ch1_crest_factor',
#         'ch1_shape_factor',   'ch1_impulse_factor',
#         'ch2_kurtosis',       'ch2_crest_factor',
#         'ch2_shape_factor',   'ch2_impulse_factor',
#         'ch4_kurtosis',       'ch4_crest_factor',
#         'ch4_shape_factor',   'ch4_impulse_factor',
#     ]

#     if 'life_pct' in df.columns:
#         early_mask = df['life_pct'] <= 20
#     else:
#         t_max      = df['t_abs'].max()
#         early_mask = df['t_abs'] <= t_max * 0.2

#     early_df = df[early_mask]

#     intensity = early_df[intensity_feats].mean().mean() \
#                 if len(early_df) > 0 else 0.0

#     df['initial_fault_intensity'] = intensity
#     return df


# # ── 전체 파이프라인 ───────────────────────────────────────────
# def run_feature_engineering(data_num, is_validation=False):
#     if is_validation:
#         path = (f'../data/Validation/'
#                 f'Validation{data_num}_vibration_featured.csv')
#     else:
#         path = f'{BASE_PATH}/Train{data_num}_vibration_featured.csv'

#     df = pd.read_csv(path)

#     if 'rpm_pred' not in df.columns:
#         raise ValueError(
#             'rpm_pred 컬럼 없음. rpm_prediction.py 먼저 실행하세요.'
#         )

#     df = add_rms_norm(df)
#     df = add_front_rear_features(df)
#     df = add_vertical_axial_features(df)
#     df = add_initial_fault_intensity(df)

#     df.to_csv(path, index=False)

#     label = 'Validation' if is_validation else 'Train'
#     print(f'{label} {data_num} 완료 | shape={df.shape}')
#     return df


# # ── 실행 ─────────────────────────────────────────────────────
# all_df = {}
# for t in [1, 2, 3, 4]:
#     all_df[t] = run_feature_engineering(t, is_validation=False)

# # Validation (데이터 수령 후)
# # for v in [1, 2, ...]:
# #     run_feature_engineering(v, is_validation=True)

# # ── 초기 결함 강도 확인 ───────────────────────────────────────
# print('\n=== 초기 결함 강도 (Train별) ===')
# for t, df in all_df.items():
#     intensity = df['initial_fault_intensity'].iloc[0]
#     print(f'Train {t} | initial_fault_intensity = {intensity:.4f}')

# # ── 최종 피처 목록 ────────────────────────────────────────────
# print('\n=== 최종 피처 목록 ===')
# meta_cols = ['epoch', 'window', 't_abs', 'rpm', 'rpm_pred',
#              'ttf', 'RUL', 'life_pct']
# feat_cols = [c for c in all_df[1].columns if c not in meta_cols]
# print(f'총 {len(feat_cols)}개')
# for c in feat_cols:
#     print(f'  {c}')